## Environment prep

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import os

## Load data

In [ ]:
# Specify the path to your main folder
main_folder_path = 'c:/Users/Ayush/Desktop/New folder (2)/ESP32-WiFi-Sensing-master/datasets/03-tvat-segments'

# Initialize an empty dictionary to store lists of DataFrames for each activity
activity_dataframes = {}

# Iterate through subfolders
for activity_folder in os.listdir(main_folder_path):
    activity_folder_path = os.path.join(main_folder_path, activity_folder)

    # Check if it's a directory
    if os.path.isdir(activity_folder_path):
        # Initialize an empty list to store DataFrames for each file in the subfolder
        file_dataframes = []

        # Iterate through files in the subfolder
        for file_name in os.listdir(activity_folder_path):
            if file_name.endswith('.csv'):
                file_path = os.path.join(activity_folder_path, file_name)

                # Read the data from each CSV file into a DataFrame
                df = pd.read_csv(file_path)  # Adjust the read_csv parameters based on your file format
                file_dataframes.append(df)

        # Append the list of DataFrames for the activity to the dictionary
        activity_dataframes[activity_folder] = file_dataframes

In [ ]:
print(activity_dataframes['LL'])

## Append label to a list

In [ ]:
labels = []

### JJ

In [ ]:
label = 0
for df in activity_dataframes['JJ']:
  labels.append(label)
  df = df.drop('timestamp',axis=1,inplace=True)

In [ ]:
print(len(labels))

In [ ]:
print(activity_dataframes['JJ'][3])

### LA

In [ ]:
label = 1
for df in activity_dataframes['LA']:
  labels.append(label)
  df = df.drop('timestamp',axis=1,inplace=True)
print(activity_dataframes['LA'][3])

In [ ]:
print(len(labels))

### LL

In [ ]:
label = 2
for df in activity_dataframes['LL']:
  labels.append(label)
  df = df.drop('timestamp',axis=1,inplace=True)
print(activity_dataframes['LL'][3])

### NA

In [ ]:
label = 3
for df in activity_dataframes['NA']:
  labels.append(label)
  df = df.drop('timestamp',axis=1,inplace=True)
print(activity_dataframes['NA'][3])

### RA

In [ ]:
label = 4
for df in activity_dataframes['RA']:
  labels.append(label)
  df = df.drop('timestamp',axis=1,inplace=True)
print(activity_dataframes['RA'][3])

### RL

In [ ]:
label = 5
for df in activity_dataframes['RL']:
  labels.append(label)
  df = df.drop('timestamp',axis=1,inplace=True)
print(activity_dataframes['RL'][3])

### SO

In [ ]:
label = 6
for df in activity_dataframes['SO']:
  labels.append(label)
  df = df.drop('timestamp',axis=1,inplace=True)
print(activity_dataframes['SO'][3])

## Form data array from dataframes

In [ ]:
# Assuming activity_dataframes is the dictionary of lists of DataFrames
activity_arrays = {}

for activity, dataframes_list in activity_dataframes.items():
    # Combine DataFrames along a new axis
    activity_array = np.stack([df.values for df in dataframes_list])
    activity_arrays[activity] = activity_array

# Concatenate activity arrays along a new axis
dataset_array = np.concatenate(list(activity_arrays.values()), axis=0)

In [ ]:
print(dataset_array.shape)

In [ ]:
print(len(labels))

### CNN

In [ ]:
from sklearn.model_selection import train_test_split
labels = np.array(labels)

X_train, X_test, y_train, y_test = train_test_split(dataset_array, labels, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=42)

X_train = X_train.astype('float32')
X_train /= 255
X_val = X_val.astype('float32')
X_val /= 255
X_test = X_test.astype('float32')
X_test /= 255

In [ ]:
print(y_train)

In [ ]:
from tensorflow import keras
from tensorflow.keras.layers import Dense, Dropout,Attention

# Define the CNN model
model = keras.Sequential([
    keras.layers.Conv2D(32, 7, activation='relu', input_shape=(200, 55,1)),
    keras.layers.MaxPooling2D(2),
    keras.layers.Conv2D(64, 5, activation='relu'),
    keras.layers.MaxPooling2D(2),
    keras.layers.Flatten(),
    keras.layers.Dense(256, activation='relu'),
    #keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.25),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.25),
    keras.layers.Dense(7, activation='softmax')

])

# Compile the model
model.compile(optimizer= tf.keras.optimizers.Adam(learning_rate=0.0001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

In [ ]:

history = model.fit(X_train, y_train, epochs=100, batch_size=40,validation_data=(X_val, y_val))

### DNN

In [ ]:
import tensorflow as tf

# Define the DNN model
model = tf.keras.Sequential([
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(7, activation='softmax')
])

# Define the optimizer and loss function
optimizer = tf.keras.optimizers.SGD()
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# Compile the model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
flat = dataset_array.reshape(1176 * 200, 55)
print(flat.shape)

In [ ]:
all_labels = []
for key, dataframe_list in activity_dataframes.items():
    for dataframe in dataframe_list:
        # Append the label to the labels list for each row in the DataFrame
        for index, row in dataframe.iterrows():
            all_labels.append((key))

In [ ]:
print(all_labels)

In [ ]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
label_encoder = LabelEncoder()
all_labels = label_encoder.fit_transform(all_labels)

In [ ]:
from sklearn.model_selection import train_test_split

data = dataset_array
labels = np.array(labels)
X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=0.2, random_state=42)
y_train = np.array(y_train)
print(y_train.shape)

In [ ]:
import tensorflow as tf

flattened_data = tf.reshape(data, [-1, data.shape[1] * data.shape[2]])
one_hot_encoded_data = tf.keras.utils.to_categorical(flattened_data)
encoded_data = one_hot_encoded_data.reshape(data[0], data[1], -1)
print(encoded_data.shape)

In [ ]:
print(X_train.shape)

In [ ]:
history = model.fit(X_train, y_train, epochs=30, batch_size=128,validation_split=0.2)

## Evaluation

In [ ]:
%matplotlib inline
from sklearn.metrics import confusion_matrix, accuracy_score
import itertools
import matplotlib.pyplot as plt

In [ ]:
def plot_confusion_matrix(cm, classes,
                        normalize=False,
                        title='Confusion matrix',
                        cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    print(cm)

    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, cm[i, j],
            horizontalalignment="center",
            color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

In [ ]:
predictions = model.predict(X_test)

rounded_predictions = np.argmax(predictions, axis=-1)

cm_plot_labels = ['JJ', 'LA', 'LL', 'NA', 'RA', 'RL', 'SO']
cm = confusion_matrix(y_true=y_test, y_pred=rounded_predictions)
accuracy = accuracy_score(y_test, rounded_predictions) * 100
plot_confusion_matrix(cm=cm, classes=cm_plot_labels, title=f'Confusion Matrix: CNN. Accuracy {accuracy:.02f}%')
print("Per class: ", cm.diagonal()/cm.sum(axis=1))